In [17]:
import sys
from pathlib import Path
 
try:
    ROOT = Path(__file__).parent          # running as: python build_artifacts.py
except NameError:
    ROOT = Path.cwd()                     # running inside Jupyter / Colab (no __file__)
sys.path.insert(0, str(ROOT))             # so "from features import ..." works from a notebook
 
import importlib
 
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
 
import features
importlib.reload(features)   # notebooks cache imported modules; this picks up an updated features.py
from features import (TARGET, FEATS, FEATS_DIRECT, WEATHER,
                      add_features, climatology, find_data, weather_for)

In [18]:
DATA = find_data(ROOT)
(ROOT / "artifacts").mkdir(exist_ok=True)
print("Using data file:", DATA)
TRAIN_END = "2024-12-31"

Using data file: c:\Users\Asus\OneDrive\Desktop\Sem 10 September 2026\Final Year Project 2\dataset\malaysia_daily_merged_2018_2026.csv


In [19]:
def make_xgb():
    return XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.02, subsample=0.7,
                        colsample_bytree=0.7, random_state=42, n_jobs=1)
 
 
def fit_detrended(train):
    lin = LinearRegression().fit(train[["days_since_start"]], train[TARGET])
    resid = train[TARGET] - lin.predict(train[["days_since_start"]])
    xgb = make_xgb().fit(train[FEATS], resid)
    return lin, xgb
 
 
def predict_detrended(lin, xgb, X):
    return lin.predict(X[["days_since_start"]]) + xgb.predict(X[FEATS])
 
 
hist = pd.read_csv(DATA, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
hist_f = add_features(hist)

In [20]:
# ---------------- backtest: train 2018-2024, forecast Jan 2025 -> Jun 2026 ----------------
train = hist_f[hist_f["date"] <= TRAIN_END]
test = hist_f[hist_f["date"] > TRAIN_END]
test_typical = add_features(weather_for(test["date"], climatology(hist[hist["date"] <= TRAIN_END])))
 
rows, preds = [], {}
direct = make_xgb().fit(train[FEATS_DIRECT], train[TARGET])
lin_bt, xgb_bt = fit_detrended(train)
for method in ["Direct (XGBoost + time index)", "Detrended (linear trend + XGBoost)"]:
    for label, X in [("actual conditions", test), ("typical conditions", test_typical)]:
        p = (direct.predict(X[FEATS_DIRECT]) if method.startswith("Direct")
             else predict_detrended(lin_bt, xgb_bt, X))
        e = test[TARGET].values - p
        rows.append({"Method": method, "Environmental input": label,
                     "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()),
                     "MAPE %": (np.abs(e) / test[TARGET].values).mean() * 100,
                     "Bias": e.mean()})
        preds[(method, label)] = p
backtest = pd.DataFrame(rows).round(3)
print(backtest.to_string(index=False))
 
final_pred_bt = preds[("Detrended (linear trend + XGBoost)", "typical conditions")]
resid_bt = test[TARGET].values - final_pred_bt
lo, hi = np.quantile(resid_bt, [0.05, 0.95])
pd.DataFrame({"date": test["date"], "actual": test[TARGET].values,
              "predicted": final_pred_bt}).to_csv(ROOT / "artifacts" / "backtest.csv", index=False)

                            Method Environmental input   MAE  RMSE  MAPE %   Bias
     Direct (XGBoost + time index)   actual conditions 0.551 0.691   4.382  0.323
     Direct (XGBoost + time index)  typical conditions 0.512 0.637   4.067  0.306
Detrended (linear trend + XGBoost)   actual conditions 0.433 0.527   3.804 -0.203
Detrended (linear trend + XGBoost)  typical conditions 0.384 0.454   3.360 -0.137


In [22]:
# ---------------- final model on ALL data ----------------
lin, xgb = fit_detrended(hist_f)
clim = climatology(hist)
importance = pd.Series(xgb.feature_importances_, index=FEATS).sort_values(ascending=False)
 
bundle = {
    "linear": lin, "xgb": xgb, "feats": FEATS, "clim": clim,
    "lo": float(lo), "hi": float(hi),
    "backtest": backtest, "importance": importance,
    "trained_until": str(hist["date"].max().date()),
    "trend_per_year": float(lin.coef_[0] * 365.25),
}
joblib.dump(bundle, ROOT / "artifacts" / "model_bundle.joblib")
print(f"\nSaved artifacts. Trend: {bundle['trend_per_year']:+.3f} kWh per household per day per year | "
      f"90% band: {lo:+.2f} / {hi:+.2f}")
 


Saved artifacts. Trend: +0.262 kWh per household per day per year | 90% band: -0.76 / +0.59
